# 🚀 AWS SageMaker Domain Setup — Complete CLI Guide

> **Companion notebook for the YouTube tutorial:**  
> *'AWS SageMaker Domain Setup — What Nobody Tells You (Company Email Fix + IAM Errors Solved)'*

---

## ⚠️ Common Sticking Points (Watch These!)

| # | Where You'll Get Stuck | Fix |
|---|---|---|
| 1 | Quick Setup fails with company email | Use Standard Setup + IAM auth |
| 2 | `sts:AssumeRole` ValidationException | Trust policy has a Condition block |
| 3 | S3 URI field showing 'Required' in red | Two separate S3 fields need filling |
| 4 | `LimitExceeded: PoliciesPerRole: 10` | Role already has 10 policies — use existing role |
| 5 | Same error after re-submitting | 'Create new role' selected instead of 'Use existing role' |
| 6 | Can't find which role is attached | Run describe-domain command in this notebook |

---

## 📋 Prerequisites
- AWS Account (company email is fine)
- AWS CLI installed OR use AWS CloudShell (recommended)
- Region: change `REGION` variable below to your region

---
## 0. Set Your Variables First
**Edit these before running anything else**

In [ ]:
# ============================================
# EDIT THESE VARIABLES BEFORE RUNNING
# ============================================

REGION = "ap-southeast-2"          # Change to your AWS region
ROLE_NAME = "SageMakerExecutionRole"  # Name for your execution role
DOMAIN_NAME = "my-sagemaker-domain"   # Name for your domain
USER_NAME = "sagemaker-user"          # Your IAM username

print(f"Region:      {REGION}")
print(f"Role:        {ROLE_NAME}")
print(f"Domain:      {DOMAIN_NAME}")
print(f"User:        {USER_NAME}")

---
## Step 1 — Get Your Account ID

In [ ]:
%%bash
aws sts get-caller-identity

---
## Step 2 — Create IAM User
⚠️ **Stuck here?** Company email accounts don't support IAM Identity Center — use IAM user instead

In [ ]:
%%bash
# Create IAM user
aws iam create-user --user-name sagemaker-user

# Attach SageMaker permissions to user
aws iam attach-user-policy \
  --user-name sagemaker-user \
  --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess

aws iam attach-user-policy \
  --user-name sagemaker-user \
  --policy-arn arn:aws:iam::aws:policy/AmazonS3FullAccess

aws iam attach-user-policy \
  --user-name sagemaker-user \
  --policy-arn arn:aws:iam::aws:policy/IAMFullAccess

echo "✅ IAM user created and policies attached"

---
## Step 3 — Create IAM Execution Role
⚠️ **Stuck here?** The trust policy MUST NOT have a `Condition` block during domain creation — it causes `sts:AssumeRole` to fail every time

In [ ]:
%%bash
# Create role with CORRECT trust policy (no Condition block)
aws iam create-role \
  --role-name SageMakerExecutionRole \
  --assume-role-policy-document '{
    "Version": "2012-10-17",
    "Statement": [{
      "Sid": "AssumeRole",
      "Effect": "Allow",
      "Principal": {
        "Service": "sagemaker.amazonaws.com"
      },
      "Action": "sts:AssumeRole"
    }]
  }'

echo "✅ Role created"

In [ ]:
%%bash
# Attach permissions to the role
aws iam attach-role-policy \
  --role-name SageMakerExecutionRole \
  --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess
echo "✅ SageMaker Full Access attached"

aws iam attach-role-policy \
  --role-name SageMakerExecutionRole \
  --policy-arn arn:aws:iam::aws:policy/AmazonS3FullAccess
echo "✅ S3 Full Access attached"

In [ ]:
%%bash
ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)

# Give IAM user permission to pass this role to SageMaker
aws iam put-user-policy \
  --user-name sagemaker-user \
  --policy-name PassRolePolicy \
  --policy-document "{
    \"Version\": \"2012-10-17\",
    \"Statement\": [{
      \"Effect\": \"Allow\",
      \"Action\": [\"iam:PassRole\", \"iam:GetRole\"],
      \"Resource\": \"arn:aws:iam::${ACCOUNT_ID}:role/SageMakerExecutionRole\"
    }]
  }"

echo "✅ PassRole permission set"

---
## Step 4 — Fix Trust Policy (If You Already Have a Role)
⚠️ **Stuck here?** If SageMaker auto-created a role during domain setup and it has a `Condition` block — this is what causes the `ValidationException: sts:AssumeRole` error

In [ ]:
%%bash
# Check current trust policy on your role
# Replace YOUR-ROLE-NAME with your actual role name
aws iam get-role \
  --role-name YOUR-ROLE-NAME \
  --query "Role.AssumeRolePolicyDocument"

In [ ]:
%%bash
# Fix the trust policy — remove Condition block
# Replace YOUR-ROLE-NAME with your actual role name
aws iam update-assume-role-policy \
  --role-name YOUR-ROLE-NAME \
  --policy-document '{
    "Version": "2012-10-17",
    "Statement": [{
      "Sid": "AssumeRole",
      "Effect": "Allow",
      "Principal": {
        "Service": "sagemaker.amazonaws.com"
      },
      "Action": "sts:AssumeRole"
    }]
  }' && echo "✅ Trust policy fixed — no Condition block"

---
## Step 5 — Find Your Domain and Role
⚠️ **Stuck here?** Multiple failed submits create multiple orphaned roles — use these commands to find the right one

In [ ]:
%%bash
# List all SageMaker domains
aws sagemaker list-domains \
  --region ap-southeast-2 \
  --query "Domains[*].{ID:DomainId, Name:DomainName, Status:Status}" \
  --output table

In [ ]:
%%bash
# Find the exact role attached to your domain
DOMAIN_ID=$(aws sagemaker list-domains \
  --region ap-southeast-2 \
  --query "Domains[0].DomainId" \
  --output text)

echo "Domain ID: $DOMAIN_ID"

aws sagemaker describe-domain \
  --domain-id $DOMAIN_ID \
  --region ap-southeast-2 \
  --query "DefaultUserSettings.ExecutionRole" \
  --output text

In [ ]:
%%bash
# List all SageMaker related roles (to clean up orphaned ones)
aws iam list-roles \
  --query "Roles[?contains(RoleName, 'agemaker')].{Name:RoleName, Created:CreateDate}" \
  --output table

---
## Step 6 — Verify Role Policies
⚠️ **Stuck here?** `LimitExceeded: PoliciesPerRole: 10` means too many policies — check what's attached first before adding more

In [ ]:
%%bash
# Check everything on your role in one shot
# Replace YOUR-ROLE-NAME below

echo "=== TRUST POLICY ==="
aws iam get-role \
  --role-name YOUR-ROLE-NAME \
  --query "Role.AssumeRolePolicyDocument"

echo ""
echo "=== ATTACHED POLICIES ==="
aws iam list-attached-role-policies \
  --role-name YOUR-ROLE-NAME \
  --query "AttachedPolicies[*].PolicyName" \
  --output table

echo ""
echo "=== INLINE POLICIES ==="
aws iam list-role-policies \
  --role-name YOUR-ROLE-NAME

---
## Step 7 — Create SageMaker Domain via CLI
Use this if the Console keeps failing

In [ ]:
%%bash
ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
REGION="ap-southeast-2"
ROLE_NAME="SageMakerExecutionRole"

# Get default VPC
VPC_ID=$(aws ec2 describe-vpcs \
  --filters "Name=isDefault,Values=true" \
  --query "Vpcs[0].VpcId" --output text)

# Get first subnet
SUBNET_ID=$(aws ec2 describe-subnets \
  --filters "Name=vpc-id,Values=$VPC_ID" \
  --query "Subnets[0].SubnetId" --output text)

echo "Account:  $ACCOUNT_ID"
echo "VPC:      $VPC_ID"
echo "Subnet:   $SUBNET_ID"

# Create domain
aws sagemaker create-domain \
  --domain-name "my-sagemaker-domain" \
  --auth-mode IAM \
  --default-user-settings "{\"ExecutionRole\": \"arn:aws:iam::${ACCOUNT_ID}:role/${ROLE_NAME}\"}" \
  --vpc-id $VPC_ID \
  --subnet-ids $SUBNET_ID \
  --region $REGION

echo "✅ Domain creation started"

---
## Step 8 — Create User Profile and Access Studio

In [ ]:
%%bash
ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
REGION="ap-southeast-2"
ROLE_NAME="SageMakerExecutionRole"

# Get domain ID
DOMAIN_ID=$(aws sagemaker list-domains \
  --region $REGION \
  --query "Domains[0].DomainId" \
  --output text)

echo "Domain ID: $DOMAIN_ID"

# Create user profile
aws sagemaker create-user-profile \
  --domain-id $DOMAIN_ID \
  --user-profile-name sagemaker-user \
  --user-settings "{\"ExecutionRole\": \"arn:aws:iam::${ACCOUNT_ID}:role/${ROLE_NAME}\"}" \
  --region $REGION

echo "✅ User profile created"

In [ ]:
%%bash
REGION="ap-southeast-2"

# Get domain ID
DOMAIN_ID=$(aws sagemaker list-domains \
  --region $REGION \
  --query "Domains[0].DomainId" \
  --output text)

# Generate Studio login URL (valid for 5 minutes)
aws sagemaker create-presigned-domain-url \
  --domain-id $DOMAIN_ID \
  --user-profile-name sagemaker-user \
  --expires-in-seconds 300 \
  --region $REGION \
  --query "AuthorizedUrl" \
  --output text

echo ""
echo "✅ Copy the URL above and paste into your browser"

---
## 💰 Cost Management — Avoid Surprise Bills

In [ ]:
%%bash
# Check all running endpoints (these charge 24/7)
aws sagemaker list-endpoints \
  --region ap-southeast-2 \
  --query "Endpoints[*].{Name:EndpointName, Status:EndpointStatus}" \
  --output table

In [ ]:
%%bash
# Delete an endpoint when done testing
# Replace YOUR-ENDPOINT-NAME below
aws sagemaker delete-endpoint \
  --endpoint-name YOUR-ENDPOINT-NAME \
  --region ap-southeast-2

echo "✅ Endpoint deleted — no more charges"

In [ ]:
%%bash
ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)

# Check S3 bucket size
aws s3 ls s3://sagemaker-ap-southeast-2-${ACCOUNT_ID} \
  --recursive \
  --human-readable \
  --summarize | tail -2

---
## 🧹 Cleanup Orphaned Roles
⚠️ Each failed domain setup creates a new orphaned role — clean them up

In [ ]:
%%bash
# List all SageMaker execution roles with creation dates
aws iam list-roles \
  --query "Roles[?contains(RoleName, 'ExecutionRole')].{Name:RoleName, Created:CreateDate}" \
  --output table

In [ ]:
%%bash
# Delete an orphaned role (detach policies first)
# Replace OLD-ROLE-NAME with the role you want to delete

OLD_ROLE="OLD-ROLE-NAME"

# Detach all policies first
for policy in $(aws iam list-attached-role-policies \
  --role-name $OLD_ROLE \
  --query "AttachedPolicies[*].PolicyArn" \
  --output text); do
  aws iam detach-role-policy \
    --role-name $OLD_ROLE \
    --policy-arn $policy
  echo "Detached: $policy"
done

# Now delete the role
aws iam delete-role --role-name $OLD_ROLE
echo "✅ Orphaned role deleted: $OLD_ROLE"

---
## ✅ Summary Checklist

```
[ ] IAM user created
[ ] IAM user has SageMaker + S3 + IAM policies
[ ] IAM execution role created (SageMakerExecutionRole)
[ ] Trust policy has NO Condition block
[ ] Role has AmazonSageMakerFullAccess + AmazonS3FullAccess
[ ] User has PassRole permission on the execution role
[ ] Domain setup Step 2 → "Use an existing role" selected
[ ] Domain created successfully
[ ] User profile created
[ ] Studio opens via presigned URL
[ ] Billing alert set at $10
```

---

## 💡 Golden Rules to Avoid Costs

```
✅ Always shut down Studio after use → File → Shut Down All
✅ Delete endpoints immediately after testing
✅ Use ml.t3.medium for learning ($0.05/hr)
✅ Use spot instances for training (saves 90%)
✅ Set a $10 billing alert in AWS Budgets
❌ Never leave an endpoint running overnight
❌ Never use GPU instances unless needed
```